# Section 3: Spark RDD MapReduce Optimisation

This section implements the same main task as Section 2 using Spark RDDs.
The goal is to optimise the most time consuming steps identified in Section 2
using the MapReduce programming paradigm.

The same cleaning and recoding logic from Section 2 is reused so the
outputs can be compared fairly on the same dataset.

## Note on SparkSession Import

`SparkSession` is imported solely as the standard entry point to
initialise Spark. All data processing in this notebook uses the
**Spark Core RDD API exclusively** via `spark.sparkContext`.

No Spark SQL queries, DataFrame operations, or Dataset APIs are
used anywhere in this notebook. Every transformation and action
is performed using RDD operations:
- `sc.textFile()` — load data as RDD
- `rdd.map()` — Map transformations
- `rdd.filter()` — filter transformations  
- `rdd.reduceByKey()` — Reduce aggregation
- `rdd.collect()` — trigger execution

In [ ]:
!pip install pyspark -q

import time
import csv
import os
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChicagoCrimeSection3") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

file_path = "Crimes_-_2001_to_Present.csv"
print("Exists:", os.path.exists(file_path))

# Section 2 baseline times
section2_load_time = 18.65
section2_clean_time = 6.92
section2_recode_time = 1.26
section2_agg_time = 0.33
section2_total_time = round(18.65 + 6.92 + 1.26 + 0.33, 2)
print(f"Section 2 total baseline time: {section2_total_time:.2f} seconds")

Exists: True
Section 2 total baseline time: 27.16 seconds


## Step 2: Most Time-Consuming Steps from Section 2

The timing results from Section 2 on the 1.42M row subset were:

| Stage | Time |
|---|---|
| Load dataset | 18.65s |
| Clean and transform | 6.92s |
| Recode crime categories | 1.26s |
| Grouped aggregation | 0.33s |
| **Total** | **27.16s** |

Loading (18.65s) is I/O bound and cannot be parallelised by MapReduce,
it is limited by disk read speed, not compute.

The two selected optimisation targets are:

1. **Clean and recode (8.18s combined)** — each row is processed
   independently with no dependency on any other row, making this a
   perfect Map operation that can be split freely across partitions
   
2. **Grouped aggregation (0.33s)** — maps directly to `reduceByKey`,
   where records sharing the same `(Community Area, Crime Category)`
   key are counted in parallel across partitions

## Step 3: Expectation

We expect Spark to outperform the pandas baseline by processing
rows in parallel across multiple cores.

Our machine has 10 logical cores, so theoretically Spark could
achieve up to 10x speedup. In practice we expect 2 to 3x due to:
- Spark startup overhead
- Python serialisation cost through Py4J
- Shuffle overhead from reduceByKey

**Expected speedup: 2x faster than Section 2 (target: ~13 seconds)**

## Step 4: Helper Functions

The helper functions below are used to keep the Spark pipeline clear and consistent with Section 2.

- `parse_csv_line` reads each CSV row safely
- `clean_main` removes invalid values and keeps the fields needed for analysis
- `recode_crime` maps detailed crime types into broader crime categories

Using the same logic as Section 2 helps make the comparison fair.

In [ ]:
def parse_csv_line(line):
    return next(csv.reader([line]))

def clean_and_recode(row):
    primary_type = str(row[0]).strip().upper()
    comm_area = str(row[1]).strip()

    if not primary_type or not comm_area:
        return None
    try:
        comm_area = int(float(comm_area))
    except ValueError:
        return None

    if primary_type in ["HOMICIDE", "ASSAULT", "BATTERY", "ROBBERY",
                        "KIDNAPPING", "INTIMIDATION", "STALKING"]:
        category = "VIOLENT CRIME"
    elif primary_type in ["CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
                          "SEX OFFENSE", "HUMAN TRAFFICKING"]:
        category = "SEXUAL / TRAFFICKING"
    elif primary_type in ["THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
                          "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
                          "DECEPTIVE PRACTICE", "ARSON"]:
        category = "PROPERTY CRIME"
    elif primary_type in ["NARCOTICS", "OTHER NARCOTIC VIOLATION"]:
        category = "DRUG OFFENSE"
    elif primary_type in ["PROSTITUTION", "GAMBLING", "LIQUOR LAW VIOLATION",
                          "PUBLIC INDECENCY", "OBSCENITY", "RITUALISM",
                          "PUBLIC PEACE VIOLATION"]:
        category = "PUBLIC ORDER"
    elif primary_type in ["WEAPONS VIOLATION", "CONCEALED CARRY LICENSE VIOLATION"]:
        category = "WEAPONS OFFENSE"
    elif primary_type == "OFFENSE INVOLVING CHILDREN":
        category = "CRIMES AGAINST CHILDREN"
    elif primary_type == "INTERFERENCE WITH PUBLIC OFFICER":
        category = "LAW ENFORCEMENT RELATED"
    elif primary_type == "OTHER OFFENSE":
        category = "MISCELLANEOUS CRIME"
    elif primary_type == "NON-CRIMINAL":
        category = "NON-CRIMINAL"
    else:
        category = "UNCLASSIFIED"

    return ((comm_area, category), 1)

print("Helper functions defined.")

Helper functions defined.


## Step 5: MapReduce Design

The Spark pipeline follows the MapReduce model.

### Map step
Each row is cleaned, the crime type is recoded, and the row is turned into a key value pair in this form:

`((community_area, crime_category), 1)`

This works well in the Map stage because each row can be handled on its own.

### Reduce step
All values with the same key are added together using `reduceByKey`. This gives the final count for each `(Community Area, Crime Category)` pair.

### Shuffle
Before the Reduce step, Spark has to move matching keys together across partitions. This is called the shuffle stage. It adds overhead, but it is necessary for grouped counting.

## Step 6: Total Pipeline Timing

This step measures the runtime of the full Spark pipeline from loading the file to producing the grouped result.

This gives an overall runtime that can be compared with the baseline from Section 2.

In [ ]:
start_total = time.time()

# Load RDD
rdd = sc.textFile(file_path)

# Remove header
header = rdd.first()
rdd = rdd.filter(lambda row: row != header)

# Parse CSV
parsed_rdd = rdd.map(parse_csv_line)

# Filter valid rows
parsed_rdd = parsed_rdd.filter(lambda cols: len(cols) > 13)

# Select Primary Type and Community Area
selected_rdd = parsed_rdd.map(lambda cols: (cols[5], cols[13]))

# MAP - clean, recode and emit key-value pairs
pair_rdd = selected_rdd.map(clean_and_recode).filter(lambda x: x is not None)

# REDUCE - sum counts by key
counts_rdd = pair_rdd.reduceByKey(lambda a, b: a + b)

# Trigger execution
result = counts_rdd.collect()
total_time = time.time() - start_total

print(f"Number of grouped keys: {len(result)}")
print(f"Total Spark pipeline time: {total_time:.2f} seconds")

Number of grouped keys: 706
Total Spark pipeline time: 15.58 seconds


## Step 7: Measure the Reduce Stage Separately

The grouped counting stage is measured separately to isolate the Reduce part of the MapReduce workflow.

Although this was not the dominant cost in Section 2, it remains an important part of the distributed implementation because it performs the final grouped aggregation by key.

In [ ]:
pair_rdd.cache()
pair_rdd.count()

start_agg = time.time()
counts_rdd = pair_rdd.reduceByKey(lambda a, b: a + b)
counts_rdd.count()
agg_time = time.time() - start_agg

print(f"Reduce stage time: {agg_time:.2f} seconds")

Reduce stage time: 3.37 seconds


## Step 8: Present MapReduce Results

The final grouped output is extracted by ordering the grouped counts in descending order and displaying the top results.

This output should match the logical result produced in Section 2, confirming that the Spark implementation preserves correctness while changing the computation model.

In [ ]:
top_counts = sorted(result, key=lambda x: -x[1])[:20]

results_df = pd.DataFrame([
    {"Community Area": k[0], "Crime Category": k[1], "Count": v}
    for k, v in top_counts
])

print("Top 20 grouped counts by (Community Area, Crime Category):")
results_df

Top 20 grouped counts by (Community Area, Crime Category):


,Community Area,Crime Category,Count
0,8,PROPERTY CRIME,39972
1,28,PROPERTY CRIME,34511
2,25,PROPERTY CRIME,32360
3,32,PROPERTY CRIME,30608
4,24,PROPERTY CRIME,28470
5,25,VIOLENT CRIME,27526
6,6,PROPERTY CRIME,24549
7,43,PROPERTY CRIME,23706
8,43,VIOLENT CRIME,18375
9,71,PROPERTY CRIME,18226


## Step 9: Timing Summary and Comparison

The timing results from Section 2 and Section 3 are compared below.
Both sections use the same 1.42M row subset making this a direct
and fair comparison. Section 2 provides the traditional non parallel
baseline, while Section 3 applies the same logic using Spark RDD MapReduce.

In [ ]:
comparison = pd.DataFrame({
    "Stage": [
        "Load dataset",
        "Clean and transform",
        "Recode crime categories",
        "Grouped aggregation",
        "TOTAL"
    ],
    "Section 2 - Pandas (1.42M rows)": [
        f"{section2_load_time:.2f}s",
        f"{section2_clean_time:.2f}s",
        f"{section2_recode_time:.2f}s",
        f"{section2_agg_time:.2f}s",
        f"{section2_total_time:.2f}s"
    ],
    "Section 3 - Spark RDD (1.42M rows)": [
        "N/A",
        "N/A",
        "N/A",
        f"{agg_time:.2f}s",
        f"{total_time:.2f}s"
    ]
})

print("Both sections run on the same 1.42M row subset.")
comparison

Both sections run on the same 1.42M row subset.


,Stage,Section 2 - Pandas (1.42M rows),Section 3 - Spark RDD (1.42M rows)
0,Load dataset,18.65s,N/A
1,Clean and transform,6.92s,N/A
2,Recode crime categories,1.26s,N/A
3,Grouped aggregation,0.33s,3.37s
4,TOTAL,27.16s,15.58s


## Step 10: Compare Section 2 and Section 3

Both sections process the same 1,420,565 row subset making this
a direct and fair comparison.

Section 2 (single threaded pandas) completed in **27.16 seconds**.
Section 3 (Spark RDD MapReduce) completed in **15.58 seconds**.

This is a **1.74x speedup** using the same data on the same machine,
demonstrating that MapReduce parallelism provides a clear performance
benefit even in local mode.

The same Community Areas and Crime Categories appear at the top of
both results, confirming the logic is fully equivalent.

## Step 11: Explain the Results

The Spark pipeline produces the same grouped output as Section 2,
confirming the MapReduce logic is correct and equivalent.

**Results:**
- Section 2 (pandas, single-threaded): 27.16 seconds
- Section 3 (Spark RDD, MapReduce): 15.58 seconds
- Speedup: 1.74x faster on the same 1.42M row dataset

**Why Spark is faster overall:**
1. **Parallel Map step** — clean, recode and key-value emission run
   simultaneously across 10 cores, each processing a partition
   independently
2. **Single pass** — clean, recode and map happen in one pass through
   the data rather than three separate sequential steps
3. **Partial aggregation** — reduceByKey aggregates within each
   partition first before combining, reducing data movement

**Why the speedup is 1.74x rather than the expected 2x:**
1. **JVM startup overhead** — launching SparkContext adds fixed cost
2. **Shuffle cost** — reduceByKey requires a shuffle stage even on
   one machine

**Why grouped aggregation is slower in Spark (3.37s vs 0.33s):**
Pandas groupby operates directly on an in memory DataFrame using
optimised C extensions with no shuffle needed. Spark reduceByKey
must shuffle data across partitions, serialising, writing and
reading back, even on one machine. The shuffle overhead outweighs
the benefit of parallelism at this scale.

## Step 12: Conclusion

This section reimplemented the Section 2 task using Spark RDDs
and the MapReduce model strictly using the Core API.

The clean/transform and recode steps were combined into a single
Map function, and grouped counting was handled by reduceByKey in
the Reduce step.

Spark completed the full pipeline in 15.58 seconds compared to
27.16 seconds for pandas, a 1.74x speedup on identical data.
This confirms that MapReduce parallelism provides a measurable
performance benefit even on a single machine in local mode, and
would scale significantly further on a distributed cluster.